# Twitch Breakout Categories — Exploration

Weeks 2–4 of the project plan: data health check → baseline charts → the four trend metrics → one insight.

**Rule from the plan:** every session here ends with at least one chart saved or one paragraph written.

In [ ]:
%matplotlib inline
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

DB = "../data/twitch.db"
conn = sqlite3.connect(DB)

def q(sql):
    return pd.read_sql_query(sql, conn, parse_dates=["poll_ts"])

import os
print("DB size (MB):", round(os.path.getsize(DB) / 1e6, 1) if os.path.exists(DB) else "no DB yet — run collector.py first")

## 1. Data health check
- How many polls landed? Any gaps > 30 min?
- Row counts per table.
- Edge cases: renamed games, offline-stream holes, duplicates.

In [ ]:
polls = q("SELECT poll_ts, COUNT(*) AS games FROM snapshots GROUP BY poll_ts ORDER BY poll_ts")
gaps = polls["poll_ts"].diff().dt.total_seconds().div(60)
print(f"polls: {len(polls)} | max gap (min): {gaps.max():.0f} | gaps>30min: {(gaps > 30).sum()}")
polls.tail(3)

## 2. Baseline: top-10 games, total viewers over time
Expect strong weekly cycles (weekend peaks). Save this chart — every later insight is framed against it.

In [ ]:
top10 = (q("""
    SELECT game_name, AVG(viewers) AS avg_v
    FROM snapshots GROUP BY game_id ORDER BY avg_v DESC LIMIT 10
"""))
ax = q(f"""
    SELECT poll_ts, game_name, viewers FROM snapshots
    WHERE game_name IN {tuple(top10.game_name)}
""").pivot_table(index="poll_ts", columns="game_name", values="viewers") \
   .resample("1h").mean().plot(figsize=(12, 5), legend=True)
ax.set_title("Top-10 game categories: viewers over time (hourly avg)")
plt.tight_layout(); plt.savefig("top10_viewers.png", dpi=150)

## 3. Daily aggregation — the base table for all metrics

In [ ]:
daily = q("""
    SELECT date(poll_ts) AS day,
           game_id,
           MAX(game_name) AS game_name,
           AVG(viewers)   AS avg_viewers,
           COUNT(DISTINCT user_id) AS streamers_live_at_poll
    FROM streams GROUP BY day, game_id
""")
# NOTE: streamer counts from COUNT(DISTINCT user_id) are an UNDERCOUNT of unique
# streamers per day (only those live at poll time). Document this in the memo.

## 4. The four metrics (Week 3)

| Metric | Definition | Why |
|---|---|---|
| Growth slope | 7d linear fit of daily avg viewers | Rising or fading? |
| Streamer-to-viewer ratio | streamers ÷ viewers | Few streamers + many viewers = under-served supply |
| New-streamer inflow | first-seen streamers per week per game | Fad vs. creator wave |
| Language mix shift | % non-English WoW | Regional trends the front page misses |

In [ ]:
# Growth slope: fit per game over trailing 7 days
import numpy as np

def growth_slope(g):
    g = g.sort_values("day").tail(7)
    if len(g) < 5:
        return np.nan
    return np.polyfit(range(len(g)), g["avg_viewers"], 1)[0]

slopes = daily.groupby("game_id").apply(growth_slope).rename("growth_slope")
# TODO: streamer-to-viewer ratio, new-streamer inflow, language mix

## 5. Breakout score (Week 4 — stretch)
`score = z(growth_slope) × under_served_ratio`

Output: ranked table → the single chart + memo. Remember the adversarial check:
is one event / one big streamer driving the signal? Use medians, exclude outliers.